In [1]:
import h5py
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

import matplotlib.colors as mcolors
from matplotlib.colors import LogNorm, PowerNorm
from mpl_toolkits.axes_grid1 import make_axes_locatable

from ipywidgets import interact, FloatSlider

In [2]:
#Requires running ConvertH5_vKonstanz.ipynb before for the same folder or the data in format *.h5 

#Enter the folder where data is in "address" (do not erase the r)
#Enter the file name that is targeted for the analysis.

#"inPLane" determines if the measurements is done inP or not, and chooses the correct Magnetic field parametrization

In [3]:
#Folder where file is. Keep the r to use raw string in Windows.
#address = Path(r"C:\Users\jose1\OneDrive\PostDoc_FZU\DATA\joseS\TFO_DSO_IX\in_Plane_H_001\FMR_2025-12-09") #TFO_DSO [001]
#address = Path(r"C:\Users\jose1\OneDrive\PostDoc_FZU\DATA\joseS\TFO_DSO_IX\in_Plane_H_1m10\FMR_2025-12-03") #TFO_DSO [1-10]
#address = Path(r"C:\Users\jose1\OneDrive\PostDoc_FZU\DATA\joseS\TFO_DSO_IX\out_Plane_H_110\FMR_2025-12-09-10") #TFO_DSO [110]
address = Path(r"C:\Users\jose1\OneDrive\PostDoc_FZU\DATA\joseS\G228N2\inP_100\FMR_2026-06-04\2026-06-04_FMR_fsw_0Ato19p5Ain81_0p5to49GHz_10dBm_18-15.h5.ma8") #TFO_DSO [110] Substrate side
#address = Path(r"C:\Users\jose1\OneDrive\PostDoc_FZU\DATA\joseS\TFO_GSO_VIII\in_Plane_H_1m10\FMR_2025-12-10") #TFO_GSO [1-10]
#address = Path(r"C:\Users\jose1\OneDrive\PostDoc_FZU\DATA\joseS\TFO_GSO_VIII\in_Plane_H_001\FMR_2025-12-11") #TFO_GSO [100]
#address = Path(r"C:\Users\jose1\OneDrive\PostDoc_FZU\DATA\joseS\TFO_GSO_VIII\out_Plane_H_110\FMR_2025-12-12") #TFO_GSO [110]
#address = Path(r"C:\Users\jose1\OneDrive\PostDoc_FZU\DATA\joseS\ml20250815b\in_Plane\FMR_2025-10-16-17") #Bulk Hematite inP
#address = Path(r"C:\Users\jose1\OneDrive\PostDoc_FZU\DATA\joseS\FEOS147\in_Plane\FMR_2025-09-24") #Hematite 330nm inP
#address = Path(r"C:\Users\jose1\OneDrive\PostDoc_FZU\DATA\joseS\FEO11x\in_Plane\FMR_2025-09-24-25") #FEOS11x inP
#address = Path(r"C:\Users\jose1\OneDrive\PostDoc_FZU\DATA\joseS\m120240308b2\in_Plane\FMR_2025-12-15") #YIG ref (Michis-Minnas) inP
#address = Path(r"C:\Users\jose1\OneDrive\PostDoc_FZU\DATA\joseS\m120240308b2\out_Plane\FMR_2025-12-18") #YIG ref (Michis-Minnas) outP

In [4]:
#Only modify the name of the target file in file_name
#file_name = "2025-12-09_FMR_fsw_0Ato19p5Ain81_0p5to50GHz_5dBm_13-02.h5" #TFO_DSO [001]
#file_name = "2025-12-03_FMR_fsw_0Ato16Ain41_0p5to50GHz_-10dBm_19-00.h5" #TFO_DSO [1-10]
#file_name = "2025-12-10_FMR_fsw_0Ato19p5Ain81_0p5to50GHz_-10dBm_11-36.h5" #TFO_DSO [110]
file_name = "2025-12-19_FMR_fsw_0Ato19p5Ain81_0p5to50GHz_-5dBm_22-05.h5" #TFO_DSO [110] substrate side -5dBm
#file_name = "2025-12-19_FMR_fsw_0Ato19p5Ain81_0p5to50GHz_-10dBm_19-21.h5" #TFO_DSO [110] substrate side -10dBm
#file_name = "2025-12-10_FMR_fsw_0Ato19p5Ain81_0p5to50GHz_-5dBm_22-28.h5" #TFO_GSO [1-10]
#file_name = "2025-12-11_FMR_fsw_0Ato19p5Ain81_0p5to50GHz_5dBm_20-35.h5" #TFO_GSO [100]
#file_name = "2025-12-12_FMR_fsw_0Ato19p5Ain81_0p5to50GHz_5dBm_11-12.h5" #TFO_GSO [110]
#file_name = "2025-10-16_FMR_fsw_0Ato19p5Ain401_1to50GHz_10dBm_19-09.h5" #Bulk_Hematite inP
#file_name = "2025-09-24_FMR_fsw_0Ato18Ain121_10to40GHz_0dBm_12-57.h5" #FEOS147 inP
#file_name = "2025-09-24_FMR_fsw_0Ato18Ain121_10to40GHz_0dBm_19-00.h5" #FEOS11x inP
#file_name = "2025-12-15_FMR_fsw_0Ato19p5Ain81_0p5to50GHz_5dBm_14-51.h5" #YIG ref (Michis-Minnas) inP
#file_name = "2025-12-18_FMR_fsw_0Ato19p5Ain81_0p5to50GHz_5dBm_14-42.h5" #YIG ref (Michis-Minnas) outP 5dBm
#file_name = "2025-12-18_FMR_fsw_0Ato19p5Ain81_0p5to50GHz_0dBm_11-57.h5" #YIG ref (Michis-Minnas) outP 0dBm

In [5]:
file = address/file_name

In [6]:
inPlane = False

In [7]:
#Constants
mu_0=4*np.pi*1e-7

In [8]:
#Material parameters
#Usable with omega_Kalinilos()
Msat_YIG = 139.6e3 #A/m
Aex_YIG = 4.0e-12 #J/m
gamma_YIG = 1.822123739e11
t_YIG = 105*1e-9

#Usable with calculate_modes()
HD_Hema = 2.3 #T
HK_Hema = 120*1e-6 #T
Hex_Hema = 1000 #T

In [9]:
def calcField(A, inP): #A current in amper
    #Field for 20mmGap, from Manual (inPlane)
    field1 = 0.01018 + 0.05859*A - 0.00118*(A**2) + 2.1658E-4*(A**3) - 1.77652E-5*(A**4) + 6.35589E-7*(A**5) -1.15951E-8*(A**6) + 1.06504E-10*(A**7) - 3.91644E-13*(A**8)
    #Cal_12152025_Using_ml20240308b2_dS11_5dBm (inPlane)
    field2 = 0.00675 + 0.05052*A + 6.69033E-4*(A**2) - 4.38998E-4*(A**3) + 1.11737E-4*(A**4) - 1.45078E-5*(A**5) + 1.00456E-6*(A**6) - 3.52383E-8*(A**7) + 4.88931E-10*(A**8)
    #Cal_18122025_Using_ml20240308b2_dS11_0dBm (outPlane)
    field_outP = -4.75612131323894e-10*A**8 + 5.53240901595132e-8*A**7 - 2.44397032127303e-6*A**6 + 5.44573750852832e-5*A**5 - 0.000670050456046096*A**4 + 0.00455904626621348*A**3 - 0.0156901329345105*A**2 + 0.0762165481027225*A + 0.0164075279665365

    field_inP = (field1 + field2)/2
    
    if inP == True:
        field = field_inP
    else:
        field = field_outP
    return field #field in T

In [10]:
def plot_fvsH_matplt(field, freq, S11, S12, S21, S22,
                 zmin_reflect=None, zmax_reflect=None,    # for S11, S22
                 zmin_trans=None, zmax_trans=None):        # for S12, S21
    
    #field = field*1e3 #mTf
    sorted_indices = np.argsort(field)
    sorted_field = field[sorted_indices]
    fig, axes = plt.subplots(2, 2, figsize=(12, 10), sharex=True, sharey=True)

    # To unify 2D grid structure for imshow
    extent = [sorted_field.min(), sorted_field.max(), freq.min(), freq.max()]

    # ---- Row 1 ----
    im11 = axes[0, 0].imshow(S11,
                             extent=extent,
                             aspect='auto',
                             origin='lower',
                             vmin=zmin_reflect, vmax=zmax_reflect)
    axes[0, 0].set_title("S11")

    im12 = axes[1, 1].imshow(S12,
                             extent=extent,
                             aspect='auto',
                             origin='lower',
                             vmin=zmin_trans, vmax=zmax_trans)
    axes[0, 1].set_title("S22")

    # ---- Row 2 ----
    im21 = axes[1, 0].imshow(S21,
                             extent=extent,
                             aspect='auto',
                             origin='lower',
                             vmin=zmin_trans, vmax=zmax_trans)
    axes[1, 0].set_title("S21")

    im22 = axes[0, 1].imshow(S22,
                             extent=extent,
                             aspect='auto',
                             origin='lower',
                             vmin=zmin_reflect, vmax=zmax_reflect)
    axes[1, 1].set_title("S12")

    # Shared labels
    axes[1, 0].set_xlabel("Field")
    axes[1, 1].set_xlabel("Field")
    axes[0, 0].set_ylabel("Frequency")
    axes[1, 0].set_ylabel("Frequency")

    # ---- Colorbars using make_axes_locatable ----
    # Reflection (S11 + S22)
    divider = make_axes_locatable(axes[0,1])
    cax_reflect = divider.append_axes("right", size="3%", pad=0.1)
    fig.colorbar(im11, cax=cax_reflect, label="Reflection magnitude")

    # Transmission (S12 + S21)
    divider = make_axes_locatable(axes[1,1])
    cax_trans = divider.append_axes("right", size="3%", pad=0.1)
    fig.colorbar(im12, cax=cax_trans, label="Transmission magnitude")

    fig.tight_layout()
    #plt.show()
    return fig, axes


In [11]:
def plot_fvsH_onlyij(field, freq, S12, S21, which,
                 zmin_trans=None, zmax_trans=None):        # for S12, S21
    
    #field = field*1e3 #mT
    #sorted_indices = np.argsort(field)
    #sorted_field = field[sorted_indices]
    sorted_field = field 
    #S12 = S12[sorted_indices]
    #S21 = S21[sorted_indices]
    fig, axes = plt.subplots(1, 1, figsize=(10, 5), sharex=True, sharey=True)

    # To unify 2D grid structure for imshow
    extent = [sorted_field.max(), sorted_field.min(), freq.min(), freq.max()]

    # ---- Row 1 ----
    if which == '21':
        im11 = axes.imshow(S21,
                                 extent=extent,
                                 aspect='auto',
                                 origin='lower',
                                 vmin=zmin_trans, vmax=zmax_trans)
        axes.set_title("S21", fontsize=20)
    else:
        im11 = axes.imshow(S12,
                                 extent=extent,
                                 aspect='auto',
                                 origin='lower',
                                 vmin=zmin_trans, vmax=zmax_trans)
        axes.set_title("S12", fontsize=20)


    # Shared labels
    axes.set_xlabel("Applied magnetic Field (T)", fontsize=16)
    axes.set_ylabel("Frequency (GHz)", fontsize=16)

    # ---- Colorbars using make_axes_locatable ----
    # Transmission (S12 + S21)
    divider = make_axes_locatable(axes)
    cax_trans = divider.append_axes("right", size="3%", pad=0.1)
    cbar = fig.colorbar(im11, cax=cax_trans, label="Transmission magnitude")
    cbar.ax.tick_params(labelsize=16) 

    fig.tight_layout()
    #plt.show()
    return fig, axes

In [12]:
def omega_Kalinilos(k, gamma, M_s, B_u, B_s, A, T, B_0, kind="SW", n=0): #in GHz, Ms in A/m, B in T
    H0 = B_0/mu_0 #A/m
    Hu = B_u/mu_0 #A/m 
    Hs = B_s/mu_0 #A/m
    
    Lambda = np.sqrt(2*A / (mu_0 *M_s**2))
    k_n = np.sqrt(k**2 + (n*np.pi/T)**2)
    rat = k**2 / k_n**2
    kron_n = (n == 0)
    P_nn = rat*(1 - (2/(1+kron_n))*rat*(1-(-1)**n *np.exp(-k*T))/(k*T))
    
    if kind == "BVW":
        return gamma * mu_0 * np.sqrt(
            (H0 + M_s* Lambda**2 * k_n**2) *\
                (H0 - Hu - (2)**n*Hs + M_s*Lambda**2 *k_n**2 + M_s*(1-P_nn))
        )*(1e-9/(2*np.pi))
    if kind == "DE":
        return gamma * mu_0 * np.sqrt(
            (H0 + M_s * Lambda**2 *k_n**2 + M_s*P_nn) *\
                (H0 - Hu - (2)**n*Hs + M_s*Lambda**2 *k_n**2 + M_s*(1-P_nn))
        )*(1e-9/(2*np.pi))
    if kind == "outP":
        return gamma * mu_0 * np.sqrt(
            (H0 - (M_s-Hu) ) *\
                (H0 - (M_s-Hu))
        )*(1e-9/(2*np.pi))
    else:
        print("No valid spin wave kind specified in dispersion. "
              "Please use BVW, SW or outP.")
        return k*0

In [13]:
def calculate_modes(H, k, HD, HK, Hex):
    #all fields in T, k in rad/m 
    #frequencies in GHz
    gamma_Hema = 29 #GHz/T
    aex = 1.7E-10 #m, Wang2023_Supp
    MsRHex = 0.013/(4*np.pi) #ElKanj2023
    
    fmr = gamma_Hema * np.sqrt(H*(H + HD) + HK*Hex)+0*k
    sswm = gamma_Hema * np.sqrt(H*(H + HD) + HK*Hex + (Hex*aex*k)*(Hex*aex*k))
    Kang_bulk_klln = gamma_Hema *np.sqrt( H*(H + HD) + HK*Hex + (MsRHex)*(H+HD)*(H+HD)+((Hex*aex*k)*(Hex*aex*k)))
    Kang_sw = gamma_Hema*(1/2) *( H*(H + HD) + HK*Hex + (Hex*aex*k)*(Hex*aex*k) )/((H + HD)) + gamma_Hema*(1/2)*(1+MsRHex)*(H+HD)
    modes={
        'FMR': fmr,
        'SSWM1': sswm,
        'Bulk//n': Kang_bulk_klln,
        'SW': Kang_sw
    }
    return modes

In [14]:
def paramagnetic_resonace(H, m, Ha):
    f = m * np.sqrt(H*H + Ha*Ha) 
    return f

In [15]:
def f_vs_H_manipulation_ij(Derivative=True, z_R=200, z_T=20, ij='21', m_para=28, Ba=0, m2_para=28, Ba2=0, Bu=0, Bs=0, BK_Hema=0, BD_Hema=0, Bex_Hema=1000):
    Ba_T = Ba/1000 #T
    Ba2_T = Ba2/1000 #T
    Bu_T = Bu/1000 #T
    Bs_T = Bs/1000 #T
    BK_T = BK_Hema*1e-6 #T
    
    colors = ['magenta', 'cyan', 'red', 'blue', 'hotpink']
    if Derivative:
        fig_manip, axes = plot_fvsH_onlyij(fields, freqs, D_dSij_dH["D_dS12M_DH_R"].T, D_dSij_dH["D_dS21M_DH_R"].T, ij,
                                           zmin_trans=-z_T, zmax_trans=z_T)
    else:
        fig_manip, axes = plot_fvsH_onlyij(fields, freqs, abs(dS_matrix["dS12"].T), abs(dS_matrix["dS21"].T), ij,
                                            zmin_trans=-z_T, zmax_trans=z_T)

    # To add a trace
    def add_trace(plt, axes, func, color, tlt):
        axes.plot(plot_fields, func, lw=2, color=color, label=tlt)
        axes.legend(loc="upper left", fontsize=14)
        axes.xaxis
    
    add_trace(fig_manip, axes, paramagnetic_resonace(abs(plot_fields),m_para, Ba_T), "blue", f'Paramagnetic')
    #add_trace(fig_manip, axes, paramagnetic_resonace(abs(plot_fields),m2_para, Ba2_T), "cyan", f'Paramagnetic')
    #add_trace(fig_manip, axes, omega_Kalinilos(0.001e6, gamma_YIG, Msat_YIG, 0, 0,  Aex_YIG, 500*1e-9, abs(plot_fields), kind="DE", n=0), 'cyan','YIG_FMR')
    add_trace(fig_manip, axes, omega_Kalinilos(0.001e6, gamma_YIG, Bs_T/mu_0, Bu_T, 0,  Aex_YIG, 500*1e-9, abs(plot_fields), kind="outP", n=0), 'red','Adjustment_FMR')
    add_trace(fig_manip, axes, calculate_modes(abs(plot_fields), 1e-9, BD_Hema, BK_T, Bex_Hema)['FMR'], "cyan", f'EasyP_AFM')
    

    axes.set_xlim(0, fields.max())
    axes.set_ylim(freqs.min(),freqs.max())
    #axes.set_xlim(0, 0.15)
    #axes.set_ylim(freqs.min(), 15)

    axes.tick_params(axis="both", labelsize=16)
    
    fig_manip.canvas.draw()
    plt.show()

In [16]:
# Open the HDF5 file
with h5py.File(file, "r") as f:
    #NumPy arrays
    S11R = f["/data/VNA d_S11R"][:]
    S11I = f["/data/VNA d_S11I"][:]
    S21R = f["/data/VNA d_S21R"][:]
    S21I = f["/data/VNA d_S21I"][:]
    S12R = f["/data/VNA d_S12R"][:]
    S12I = f["/data/VNA d_S12I"][:]
    S22R = f["/data/VNA d_S22R"][:]
    S22I = f["/data/VNA d_S22I"][:]
    
    freqs = f["/data/VNA frequency"][0, :]*1e-9 #GHz
    currs = f["/data/magnet current"][:] #A
    power = f["/data/VNA power"][0]

#Bulding dS matrix
dS_matrix = {"dS11": S11R + 1j * S11I, "dS21": S21R + 1j * S21I,
                   "dS22": S22R + 1j * S22I, "dS12": S12R + 1j * S12I,}

In [17]:
#Determine field
fields = calcField(currs, inPlane) #T
field_pts = len(fields)
#Fields for traces
plot_fields = np.linspace(fields.min(), fields.max(), 100) #T

In [18]:
#Calculate derivatives
dS21_dH = np.gradient(dS_matrix["dS21"], fields, axis=0)
dS21_M_dH = np.gradient(abs(dS_matrix["dS21"]), fields, axis=0)
dS12_dH = np.gradient(dS_matrix["dS12"], fields, axis=0)
dS12_M_dH = np.gradient(abs(dS_matrix["dS12"]), fields, axis=0)

D_dSij_dH ={
        "D_dS21_DH_M": abs(np.array(dS21_dH)), "D_dS21M_DH_M": abs(np.array(dS21_M_dH)),
        "D_dS21M_DH_R": np.array(dS21_M_dH).real,
        "D_dS12_DH_M": abs(np.array(dS12_dH)), "D_dS12M_DH_M": abs(np.array(dS12_M_dH)),
        "D_dS12M_DH_R": np.array(dS12_M_dH).real
            }

In [19]:
#Interactive sliders
interact(f_vs_H_manipulation_ij,
        Derivative=[True, False],
        z_R=FloatSlider(value=0.05, min=-5, max=5, step=0.01, readout_format='.2f', description=r"Contrast ii"),
        z_T=FloatSlider(value=0.005, min=-5, max=5, step=0.01, readout_format='.2f', description=r"Contrast ij"),
        ij=['12', '21'],
        #p0=FloatSlider(value=0.015670, min=-0.1, max=0.1, step=0.00001, readout_format='.6f', description=r"mean_parameter"),
        m_para=FloatSlider(value=28.9, min=27, max=40, step=0.1, readout_format='.2f', description=r"gamma (GHz/T)"),
        Ba=FloatSlider(value=0, min=0, max=500, step=1, readout_format='.2f', description=r"Ha_para (mT)"),
        m2_para=FloatSlider(value=30, min=27, max=30, step=0.1, readout_format='.2f', description=r"gamma (GHz/T)"),
        Ba2=FloatSlider(value=200, min=0, max=500, step=1, readout_format='.2f', description=r"Ha_para (mT)"),
        Bu=FloatSlider(value=40, min=-100, max=100, step=5, readout_format='.2f', description=r"Hu (mT)"),
        Bs=FloatSlider(value=50, min=0, max=1e3, step=0.05, readout_format='.2f', description=r"Ms (mT)"),
        BK_Hema=FloatSlider(value=-100, min=-500, max=500, step=5, readout_format='.2f', description=r"HK_Hema (uT)"),
        BD_Hema=FloatSlider(value=0.8, min=1, max=10, step=0.1, readout_format='.2f', description=r"HD_Hema (T)"),
        Bex_Hema=FloatSlider(value=100, min=0, max=1500, step=10, readout_format='.2f', description=r"Hex_Hema (T)"),
);

interactive(children=(Dropdown(description='Derivative', options=(True, False), value=True), FloatSlider(value…